# 03 — Analyze Stadium & Attendance Data

Produces the visualizations for the stadium/infrastructure half of the project:

1. A map of current stadiums, sized by capacity and colored by which league(s) play there.
2. Average NWSL home attendance by team.
3. NWSL average home attendance by season, by team.

Uses the shared color theme in `plot_theme.py` so every chart in this repo reads as part of the same project.

**Inputs:** `data/processed/stadiums.csv`, `teams_clean.csv`, `games_clean.csv`
**Outputs:** figures saved to `output/`


In [ ]:
import pandas as pd
import plotly.express as px
import os

from plot_theme import style_plotly_fig, NWSL_SEQUENTIAL

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUT_DIR = os.path.join("..", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

stadiums = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "stadiums.csv"))
teams = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "teams_clean.csv"))
games = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "games_clean.csv"))

print(f"Loaded: stadiums {stadiums.shape}, teams {teams.shape}, games {games.shape}")


## Map: current stadiums by capacity and league

In [ ]:
current_stadiums = stadiums[stadiums.primary_is_current == True].copy()
current_stadiums["capacity_clean"] = current_stadiums["capacity"].fillna(10000)

fig = px.scatter_map(
    current_stadiums,
    lat="latitude",
    lon="longitude",
    hover_name="stadium_name",
    color="competition",
    size="capacity_clean",
    size_max=20,
    zoom=3.2,
    map_style="open-street-map",
)

fig = style_plotly_fig(fig)
fig.update_layout(
    width=1000,
    height=600,
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    title={
        "text": "<b>MLS & NWSL Stadiums by Capacity</b>",
        "y": 0.95,
        "x": 0.02,
        "xanchor": "left",
        "yanchor": "top",
        "font": {"size": 20, "color": "#2a2a2a", "family": "Arial, sans-serif"},
    },
)

fig.write_image(os.path.join(OUTPUT_DIR, "stadiums_map_by_capacity.png"))
fig.show()


## NWSL average home attendance by team

In [ ]:
nwsl_games = games[games.league == "nwsl"].copy()
nwsl_teams = teams[teams.league == "nwsl"].copy()

# 1. Drop missing attendance rows
att_df = nwsl_games.dropna(subset=["attendance", "home_team_id"]).copy()

# 2. Merge with nwsl_teams to get team names
att_df = att_df.merge(
    nwsl_teams[["team_id", "team_name"]],
    left_on="home_team_id", right_on="team_id", how="left",
)

# 3. Group by team_name
team_summary = att_df.groupby("team_name").agg(
    avg_attendance=("attendance", "mean"),
    games_recorded=("game_id", "count"),
).reset_index()

# 4. Plot
fig = px.bar(
    team_summary.sort_values(by="avg_attendance", ascending=True),
    x="avg_attendance",
    y="team_name",
    orientation="h",
    color="avg_attendance",
    title="<b>Average NWSL Home Attendance by Team (Excluding Missing Records)</b>",
    labels={"avg_attendance": "Average Attendance", "team_name": "Team Name"},
    hover_data={"games_recorded": True, "avg_attendance": ":.0f"},
    color_continuous_scale=NWSL_SEQUENTIAL,
)

fig = style_plotly_fig(fig)
fig.update_layout(margin={"t": 50, "b": 10, "l": 10, "r": 10}, height=500)
fig.write_image(os.path.join(OUTPUT_DIR, "nwsl_avg_attendance_by_team.png"))
fig.show()


## NWSL average home attendance by season

In [ ]:
season_team_avg = (
    att_df.groupby(["season_name", "team_name"])["attendance"]
    .mean()
    .reset_index()
)

fig = px.line(
    season_team_avg,
    x="season_name",
    y="attendance",
    color="team_name",
    title="<b>NWSL Average Home Attendance by Season</b>",
)
fig = style_plotly_fig(fig)
fig.write_image(os.path.join(OUTPUT_DIR, "nwsl_avg_attendance_by_season.png"))
fig.show()


## Team lifespans (expansion / fold years)

In [ ]:
lifespans = teams.dropna(subset=["expansion_year"])[
    ["team_name", "league", "expansion_year", "year_of_death"]
].sort_values(["league", "expansion_year"])
lifespans
